In [1]:
import csv
import re
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.preprocessing import LabelEncoder


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        holdout = candidate / 'data' / 'processed' / 'Fase2' / 'SplitDataset' / 'Split_DLLM' / 'heloc_DLLM_holdout.csv'
        if holdout.exists():
            return candidate
    raise FileNotFoundError('Impossibile trovare la radice del progetto.')


project_root = find_project_root()
holdout_path = project_root / 'data' / 'processed' / 'Fase2' / 'SplitDataset' / 'Split_DLLM' / 'heloc_DLLM_holdout.csv'
imputated_root = project_root / 'data' / 'processed' / 'Fase3' / 'Imputated_DLLM'
output_dir = project_root / 'data' / 'processed' / 'Fase3' / 'Results'
TARGET_COL = 'RiskPerformance'

print(f'Project root : {project_root}')
print(f'Holdout      : {holdout_path} (esiste: {holdout_path.exists()})')
print(f'Imputated dir: {imputated_root}')
print(f'Output dir   : {output_dir}')

Project root : /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project
Holdout      : /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase2/SplitDataset/Split_DLLM/heloc_DLLM_holdout.csv (esiste: True)
Imputated dir: /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase3/Imputated_DLLM
Output dir   : /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase3/Results


In [2]:
def load_csv_as_df(path: Path) -> pd.DataFrame:
    """Legge un CSV e converte le colonne numeriche, lasciando NaN per i campi vuoti."""
    df = pd.read_csv(path)
    return df


def prepare_Xy(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Separa features e target; converte il target in 0/1."""
    y = (df[TARGET_COL] == 'Good').astype(int)   # Good=1, Bad=0
    X = df.drop(columns=[TARGET_COL])
    # Converte eventuali colonne object residue in float (i valori categorici semantici del DLLM)
    for col in X.select_dtypes(include='object').columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')
    return X, y


def parse_train_filename(name: str) -> tuple[str, str, str]:
    """Estrae (dataset, strategia, pct) dal nome file discriminative_train."""
    m = re.match(r'^(.+?)_discriminative_train_([A-Z]+)_(\d+)\.csv$', name)
    if not m:
        raise ValueError(f'Nome file non riconosciuto: {name}')
    return m.group(1), m.group(2), m.group(3)


# Carica holdout una sola volta
df_holdout = load_csv_as_df(holdout_path)
X_holdout, y_holdout = prepare_Xy(df_holdout)
print(f'Holdout: {X_holdout.shape[0]} righe, {X_holdout.shape[1]} feature')
print(f'Distribuzione target holdout: Good={y_holdout.sum()}, Bad={(y_holdout==0).sum()}')

Holdout: 2959 righe, 23 feature
Distribuzione target holdout: Good=1420, Bad=1539


/var/folders/8g/3_yjjvpn7113xq7pfkgkcbzr0000gn/T/ipykernel_8987/544885469.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include='object').columns:


In [3]:
# Iperparametri XGBoost
XGBOOST_PARAMS = dict(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)

results = []

# Per la pipeline DLLM i file imputati sono direttamente in Imputated_DLLM/ (senza sottocartelle per metodo)
train_files = sorted(imputated_root.glob('*_discriminative_train_*.csv'))

print(f'\n=== DLLM — {len(train_files)} file ===')

for train_path in train_files:
    dataset, strategia, pct = parse_train_filename(train_path.name)

    # Carica train imputato
    df_train = load_csv_as_df(train_path)
    X_train, y_train = prepare_Xy(df_train)

    # Addestra XGBoost sul train imputato
    model = XGBClassifier(**XGBOOST_PARAMS)
    model.fit(X_train, y_train)

    # Valuta sul holdout pulito
    y_pred = model.predict(X_holdout)
    y_prob = model.predict_proba(X_holdout)[:, 1]

    acc   = accuracy_score(y_holdout, y_pred)
    auc   = roc_auc_score(y_holdout, y_prob)
    f1    = f1_score(y_holdout, y_pred, pos_label=1)

    row = {
        'imputation_method': 'DLLM',
        'dataset'          : dataset,
        'missing_strategy' : strategia,
        'missing_pct'      : int(pct),
        'train_file'       : train_path.name,
        'holdout_file'     : holdout_path.name,
        'train_rows'       : len(X_train),
        'holdout_rows'     : len(X_holdout),
        'accuracy'         : round(acc, 6),
        'roc_auc'          : round(auc, 6),
        'f1_score'         : round(f1, 6),
    }
    results.append(row)
    print(f'  {strategia:4s} {pct:>2s}% | Acc={acc:.4f} AUC={auc:.4f} F1={f1:.4f}')

print(f'\nTotale esperimenti: {len(results)}')


=== DLLM — 9 file ===


/var/folders/8g/3_yjjvpn7113xq7pfkgkcbzr0000gn/T/ipykernel_8987/544885469.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include='object').columns:


  MAR  10% | Acc=0.7202 AUC=0.7797 F1=0.7028


/var/folders/8g/3_yjjvpn7113xq7pfkgkcbzr0000gn/T/ipykernel_8987/544885469.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include='object').columns:


  MAR  25% | Acc=0.7023 AUC=0.7676 F1=0.6762
  MAR  40% | Acc=0.6803 AUC=0.7487 F1=0.6362


...

  MCAR 40% | Acc=0.7198 AUC=0.7784 F1=0.6910
  MNAR 10% | Acc=0.7107 AUC=0.7793 F1=0.6958


/var/folders/8g/3_yjjvpn7113xq7pfkgkcbzr0000gn/T/ipykernel_8987/544885469.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include='object').columns:


  MNAR 25% | Acc=0.7178 AUC=0.7797 F1=0.7124


/var/folders/8g/3_yjjvpn7113xq7pfkgkcbzr0000gn/T/ipykernel_8987/544885469.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include='object').columns:


  MNAR 40% | Acc=0.7121 AUC=0.7738 F1=0.6970

Totale esperimenti: 9


In [4]:
# Salva i risultati in CSV
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / 'discriminative_results_DLLM_xgboost.csv'

fieldnames = [
    'imputation_method', 'dataset', 'missing_strategy', 'missing_pct',
    'train_file', 'holdout_file', 'train_rows', 'holdout_rows',
    'accuracy', 'roc_auc', 'f1_score',
]
with report_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f'Report salvato: {report_path}')

# Mostra tabella riassuntiva
df_results = pd.DataFrame(results)
df_results.sort_values(['missing_strategy', 'missing_pct'])

Report salvato: /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase3/Results/discriminative_results_DLLM_xgboost.csv


,imputation_method,dataset,missing_strategy,missing_pct,train_file,holdout_file,train_rows,holdout_rows,accuracy,roc_auc,f1_score
0,DLLM,heloc_DLLM,MAR,10,heloc_DLLM_discriminative_train_MAR_10.csv,heloc_DLLM_holdout.csv,2958,2959,0.720176,0.779681,0.702800
1,DLLM,heloc_DLLM,MAR,25,heloc_DLLM_discriminative_train_MAR_25.csv,heloc_DLLM_holdout.csv,2958,2959,0.702264,0.767648,0.676222
2,DLLM,heloc_DLLM,MAR,40,heloc_DLLM_discriminative_train_MAR_40.csv,heloc_DLLM_holdout.csv,2958,2959,0.680297,0.748661,0.636154
3,DLLM,heloc_DLLM,MCAR,10,heloc_DLLM_discriminative_train_MCAR_10.csv,heloc_DLLM_holdout.csv,2958,2959,0.720514,0.782564,0.723873
4,DLLM,heloc_DLLM,MCAR,25,heloc_DLLM_discriminative_train_MCAR_25.csv,heloc_DLLM_holdout.csv,2958,2959,0.710037,0.782130,0.712851
5,DLLM,heloc_DLLM,MCAR,40,heloc_DLLM_discriminative_train_MCAR_40.csv,heloc_DLLM_holdout.csv,2958,2959,0.719838,0.778401,0.691018
6,DLLM,heloc_DLLM,MNAR,10,heloc_DLLM_discriminative_train_MNAR_10.csv,heloc_DLLM_holdout.csv,2958,2959,0.710713,0.779313,0.695807
7,DLLM,heloc_DLLM,MNAR,25,heloc_DLLM_discriminative_train_MNAR_25.csv,heloc_DLLM_holdout.csv,2958,2959,0.717810,0.779733,0.712367
8,DLLM,heloc_DLLM,MNAR,40,heloc_DLLM_discriminative_train_MNAR_40.csv,heloc_DLLM_holdout.csv,2958,2959,0.712065,0.773771,0.697013
